In [9]:
import json
import pandas as pd
import glob
import os

def build_master_dataset(raw_folder_path, output_path):
    all_rows = []
    
    file_list = glob.glob(os.path.join(raw_folder_path, "*.json"))
    
    if not file_list:
        print(f"Error: No JSON files found in {raw_folder_path}")
        return
    
    print(f"Found {len(file_list)} matches. Processing innings and deliveries...")

    for index, file in enumerate(file_list):
        with open(file) as f:
            try:
                match_data = json.load(f)
            except Exception as e:
                print(f"Skipping {file} due to error: {e}")
                continue
        
        # 1. Metadata extraction
        match_id = os.path.basename(file).replace('.json', '')
        info = match_data.get('info', {})
        winner = info.get('outcome', {}).get('winner', 'No Result')
        venue = info.get('venue', 'Unknown')
        city = info.get('city', 'Unknown')
        
        match_date = info.get('dates', ['0000-00-00'])[0]
        match_year = match_date.split('-')[0]
        
        if winner == 'No Result' or 'result' in info.get('outcome', {}):
            continue

        # 2. Iterate through Innings
        for i, innings in enumerate(match_data.get('innings', [])):
            batting_team = innings.get('team')
            is_winner = 1 if batting_team == winner else 0
            
            current_score = 0
            wickets_lost = 0
            balls_bowled = 0
            target = innings.get('target', {}).get('runs', 0)
            
            batter_scores = {} 
            
            # --- NEW: Trackers for Advanced Temporal Features ---
            recent_deliveries = [] # Rolling memory of last 18 balls
            bowler_usage = {}      # Dictionary to track balls bowled per bowler

            for over_data in innings.get('overs', []):
                over_num = over_data.get('over')
                
                for delivery in over_data.get('deliveries', []):
                    # Tracking runs
                    total_runs_on_ball = delivery['runs']['total']
                    current_score += total_runs_on_ball
                    
                    # Tracking wickets
                    wicket_on_ball = len(delivery.get('wickets', [])) if 'wickets' in delivery else 0
                    wickets_lost += wicket_on_ball
                    
                    # Logic for "Balls Remaining" and "Bowler Usage"
                    extras_info = delivery.get('extras', {})
                    is_extra_ball = 'wides' in extras_info or 'noballs' in extras_info
                    
                    if not is_extra_ball:
                        balls_bowled += 1
                        
                        # NEW: Track how many balls this bowler has bowled
                        b_name = delivery['bowler']
                        bowler_usage[b_name] = bowler_usage.get(b_name, 0) + 1

                    # Tracking "Set Batsman" Runs
                    current_batter = delivery['batter']
                    batsman_runs_on_ball = delivery['runs']['batter']
                    batter_scores[current_batter] = batter_scores.get(current_batter, 0) + batsman_runs_on_ball
                    set_batter_runs = batter_scores[current_batter]

                    # --- ADVANCED FEATURE 1: MOMENTUM (Last 18 Balls) ---
                    # Add current ball to memory, remove oldest if we exceed 18 balls
                    recent_deliveries.append({'runs': total_runs_on_ball, 'wicket': wicket_on_ball})
                    if len(recent_deliveries) > 18:
                        recent_deliveries.pop(0)
                    
                    last_18_runs = sum(d['runs'] for d in recent_deliveries)
                    last_18_wickets = sum(d['wicket'] for d in recent_deliveries)

                    # --- ADVANCED FEATURE 2: PRESSURE DELTA (RRR vs CRR) ---
                    # max(1, ...) prevents division by zero on the final ball
                    balls_left = max(1, 120 - balls_bowled)
                    runs_req = max(0, target - current_score) if target > 0 else 0
                    
                    rrr = (runs_req * 6) / balls_left
                    crr_3_overs = (last_18_runs / len(recent_deliveries)) * 6 if recent_deliveries else 0
                    pressure_delta = rrr - crr_3_overs

                    # --- ADVANCED FEATURE 3: FINISHER QUOTA ---
                    # Sum of remaining quota for bowlers who have been introduced
                    # (Assumes max 24 balls per bowler)
                    strike_balls_left = sum(max(0, 24 - b_balls) for b_balls in bowler_usage.values())

                    # Append delivery-level data
                    all_rows.append({
                        'match_id': match_id,
                        'match_year': match_year,
                        'venue': venue,
                        'city': city,
                        'batting_team': batting_team,
                        'innings': i + 1,
                        'over': over_num,
                        'ball': balls_bowled % 6 if balls_bowled % 6 != 0 else 6,
                        'batsman': current_batter,
                        'bowler': delivery['bowler'],
                        'current_score': current_score,
                        'wickets_lost': wickets_lost,
                        'balls_left': balls_left,
                        'current_batter_runs': set_batter_runs, 
                        'target': target,
                        'runs_required': runs_req,
                        'rrr': rrr,                             # NEW
                        'last_18_runs': last_18_runs,           # NEW
                        'last_18_wickets': last_18_wickets,     # NEW
                        'pressure_delta': pressure_delta,       # NEW
                        'strike_balls_rem': strike_balls_left,  # NEW
                        'is_winner': is_winner
                    })
        
        if (index + 1) % 100 == 0:
            print(f"Processed {index + 1}/{len(file_list)} matches...")

    df = pd.DataFrame(all_rows)
    df['balls_left'] = df['balls_left'].apply(lambda x: max(0, x))
    
    df.to_csv(output_path, index=False)
    print(f"\nSuccess! Master dataset saved to {output_path}")
    print(f"Total deliveries processed: {len(df)}")
    return df

if __name__ == "__main__":
    RAW_DATA_PATH = '../data'
    PROCESSED_DATA_PATH = '../data/processed/master_deliveries.csv'
    
    os.makedirs('../data/processed', exist_ok=True)
    build_master_dataset(RAW_DATA_PATH, PROCESSED_DATA_PATH)

Found 1215 matches. Processing innings and deliveries...
Processed 100/1215 matches...
Processed 300/1215 matches...
Processed 400/1215 matches...
Processed 500/1215 matches...
Processed 600/1215 matches...
Processed 700/1215 matches...
Processed 800/1215 matches...
Processed 900/1215 matches...
Processed 1000/1215 matches...
Processed 1100/1215 matches...
Processed 1200/1215 matches...

Success! Master dataset saved to ../data/processed/master_deliveries.csv
Total deliveries processed: 283963
